In [15]:
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize

AttributeError: partially initialized module 'nltk' has no attribute 'data' (most likely due to a circular import)

In [10]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import os, glob, re, sys, random, unicodedata, collections
from tqdm import tqdm
from functools import reduce



In [5]:
!pip install rank_bm25


In [7]:
from rank_bm25 import BM25Okapi, BM25L

In [13]:
import urllib.request
import tarfile
import os

url = "http://ir.dcs.gla.ac.uk/resources/test_collections/cisi/cisi.tar.gz"

# Download
urllib.request.urlretrieve(url, "cisi.tar.gz")

# Extract
with tarfile.open("cisi.tar.gz", "r:gz") as tar:
    tar.extractall("./cisi-collection")

# Verify
print(os.listdir("./cisi-collection"))

['CISI.REL', 'CISI.BLN', 'CISI.ALL', 'CISI.QRY']


/tmp/ipykernel_40342/436549623.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("./cisi-collection")


In [18]:
import importlib, sys

# Clear any bad cached imports
for mod in list(sys.modules.keys()):
    if 'nltk' in mod:
        del sys.modules[mod]

import nltk
import numpy as np
import pandas as pd
import os, random
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi, BM25L

print("All imports OK")

All imports OK


In [19]:
import sys
print(sys.version)       # confirm Python version
print(sys.path)          # check if anything weird is in the path

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']


In [20]:
"""
Evaluating Information Retrieval Models: BM25Okapi vs BM25L
===========================================================
Metrics: Mean Average Precision (MAP) and MRR@10
Dataset: CISI Collection (Glasgow University)
"""



# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATASET LOADING
# ─────────────────────────────────────────────────────────────────────────────
# Update this path to wherever the CISI collection files live on your machine.
dataset_path = "./cisi-collection"   # if in the same folder as your script

# ── Documents ────────────────────────────────────────────────────────────────
doc_set = {}
with open(dataset_path + "/CISI.ALL") as f:
    lines = ""
    for l in f.readlines():
        lines += "\n" + l.strip() if l.startswith(".") else " " + l.strip()
    lines = lines.lstrip("\n").split("\n")

doc_id = ""
doc_text = ""
for l in lines:
    if l.startswith(".I"):
        doc_id = int(l.split(" ")[1].strip()) - 1
    elif l.startswith(".X"):
        doc_set[doc_id] = doc_text.lstrip(" ")
        doc_id = ""
        doc_text = ""
    else:
        doc_text += l.strip()[3:] + " "

# ── Queries ───────────────────────────────────────────────────────────────────
with open(dataset_path + "/CISI.QRY") as f:
    lines = ""
    for l in f.readlines():
        lines += "\n" + l.strip() if l.startswith(".") else " " + l.strip()
    lines = lines.lstrip("\n").split("\n")

qry_set = {}
qry_id = ""
for l in lines:
    if l.startswith(".I"):
        qry_id = int(l.split(" ")[1].strip()) - 1
    elif l.startswith(".W"):
        qry_set[qry_id] = l.strip()[3:]
        qry_id = ""

# ── Relevance judgements ──────────────────────────────────────────────────────
rel_set = {}
with open(dataset_path + "/CISI.REL") as f:
    for l in f.readlines():
        parts = l.lstrip(" ").strip("\n").split("\t")[0].split(" ")
        qid = int(parts[0]) - 1
        did = int(parts[-1]) - 1
        rel_set.setdefault(qid, []).append(did)

# ── Quick stats ───────────────────────────────────────────────────────────────
print("Read %d documents, %d queries and %d mappings from CISI dataset"
      % (len(doc_set), len(qry_set), len(rel_set)))

number_of_rel_docs = [len(v) for v in rel_set.values()]
print("Average %.2f and %d min relevant documents per query"
      % (np.mean(number_of_rel_docs), np.min(number_of_rel_docs)))

print("Queries without relevant documents:",
      np.setdiff1d(list(qry_set.keys()), list(rel_set.keys())))

# ─────────────────────────────────────────────────────────────────────────────
# 2.  EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def mean_reciprocal_rank(bool_results, k=10):
    """
    MRR@k — reciprocal rank of the first relevant item in the top-k results.

    Args:
        bool_results : iterable of boolean relevance arrays (one per query),
                       ordered by decreasing retrieval score.
        k            : cut-off depth (default 10).

    Returns:
        float  Mean Reciprocal Rank @k across all queries.
    """
    rs = (np.atleast_1d(r[:k]).nonzero()[0] for r in bool_results)
    return np.mean([1.0 / (r[0] + 1) if r.size else 0.0 for r in rs])


def mean_average_precision(bool_results):
    """
    MAP — Mean Average Precision across all queries.

    Average Precision for a single query is the mean of precision values
    computed at each relevant document's position in the ranked list.

    Args:
        bool_results : iterable of boolean relevance arrays (one per query),
                       ordered by decreasing retrieval score.

    Returns:
        float  MAP score.
    """
    average_precisions = []
    for r in bool_results:
        r = np.asarray(r, dtype=float)
        if r.sum() == 0:               # no relevant docs for this query
            average_precisions.append(0.0)
            continue
        # Precision@i for every position i where the document is relevant
        precisions_at_relevant = []
        num_relevant_so_far = 0
        for i, rel in enumerate(r):
            if rel:
                num_relevant_so_far += 1
                precisions_at_relevant.append(num_relevant_so_far / (i + 1))
        # AP = mean of those precision values
        average_precisions.append(np.mean(precisions_at_relevant))
    return float(np.mean(average_precisions))


# ─────────────────────────────────────────────────────────────────────────────
# 3.  RETRIEVAL HELPER
# ─────────────────────────────────────────────────────────────────────────────

def results_from_query(qry_id, bm25, tokenize_fn):
    """
    Run one query through a BM25 index and return a sorted boolean mask.

    Args:
        qry_id      : integer query id
        bm25        : a fitted BM25* object
        tokenize_fn : callable  str -> list[str]  (raw or preprocessed)

    Returns:
        numpy array  1.0 where a retrieved document is relevant, else 0.0,
                     ordered from most to least relevant.
    """
    query    = qry_set[qry_id]
    rel_docs = rel_set.get(qry_id, [])

    tokenized_query          = tokenize_fn(query)
    scores                   = bm25.get_scores(tokenized_query)
    most_relevant_documents  = np.argsort(-scores)

    mask          = np.zeros(most_relevant_documents.shape)
    mask[rel_docs] = 1
    return np.take(mask, most_relevant_documents)


# ─────────────────────────────────────────────────────────────────────────────
# 4.  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

stemmer    = PorterStemmer()
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab",  quiet=True)
stop_words = nltk.corpus.stopwords.words("english")


def preprocess_string(txt, remove_stop=True, do_stem=True, to_lower=True):
    """
    Tokenise and optionally lower-case, remove stop-words, and stem.

    Args:
        txt         : raw string
        remove_stop : remove English stop-words when True
        do_stem     : apply Porter stemmer when True
        to_lower    : convert to lower-case when True

    Returns:
        list[str]  preprocessed tokens
    """
    if to_lower:
        txt = txt.lower()
    tokens = nltk.tokenize.word_tokenize(txt)
    if remove_stop:
        tokens = [tk for tk in tokens if tk not in stop_words]
    if do_stem:
        tokens = [stemmer.stem(tk) for tk in tokens]
    return tokens


# Convenience wrappers for the two tokenisation regimes used below
raw_tokenize  = lambda text: text.split(" ")
prep_tokenize = lambda text: preprocess_string(text)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  BM25Okapi — RAW (no preprocessing)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("BM25Okapi — raw corpus (no preprocessing)")
print("=" * 60)

corpus           = list(doc_set.values())
tokenized_corpus = [raw_tokenize(doc) for doc in corpus]
bm25_okapi_raw   = BM25Okapi(tokenized_corpus)

results_okapi_raw = [
    results_from_query(qid, bm25_okapi_raw, raw_tokenize)
    for qid in list(qry_set.keys())
]

mrr_okapi_raw = mean_reciprocal_rank(results_okapi_raw)
map_okapi_raw = mean_average_precision(results_okapi_raw)   # ← TO DO 2.3

print("MRR@10 : %.4f" % mrr_okapi_raw)
print("MAP    : %.4f" % map_okapi_raw)

# ─────────────────────────────────────────────────────────────────────────────
# 6.  BM25Okapi — PREPROCESSED
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("BM25Okapi — preprocessed corpus (lower + stopwords + stemming)")
print("=" * 60)

tokenized_corpus_prep = [prep_tokenize(doc) for doc in corpus]
bm25_okapi_prep       = BM25Okapi(tokenized_corpus_prep)

results_okapi_prep = [
    results_from_query(qid, bm25_okapi_prep, prep_tokenize)
    for qid in list(qry_set.keys())
]

mrr_okapi_prep = mean_reciprocal_rank(results_okapi_prep)
map_okapi_prep = mean_average_precision(results_okapi_prep)  # ← TO DO 2.4.3

print("MRR@10 : %.4f" % mrr_okapi_prep)
print("MAP    : %.4f" % map_okapi_prep)

# ─────────────────────────────────────────────────────────────────────────────
# 7.  BM25L — RAW                                              (TO DO section 3)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("BM25L — raw corpus (no preprocessing)")
print("=" * 60)

tokenized_corpus_raw = [raw_tokenize(doc) for doc in corpus]
bm25_l_raw           = BM25L(tokenized_corpus_raw)

results_l_raw = [
    results_from_query(qid, bm25_l_raw, raw_tokenize)
    for qid in list(qry_set.keys())
]

mrr_l_raw = mean_reciprocal_rank(results_l_raw)
map_l_raw = mean_average_precision(results_l_raw)

print("MRR@10 : %.4f" % mrr_l_raw)
print("MAP    : %.4f" % map_l_raw)

# ─────────────────────────────────────────────────────────────────────────────
# 8.  BM25L — PREPROCESSED                                     (TO DO section 3)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("BM25L — preprocessed corpus (lower + stopwords + stemming)")
print("=" * 60)

bm25_l_prep = BM25L(tokenized_corpus_prep)   # reuse the preprocessed corpus

results_l_prep = [
    results_from_query(qid, bm25_l_prep, prep_tokenize)
    for qid in list(qry_set.keys())
]

mrr_l_prep = mean_reciprocal_rank(results_l_prep)
map_l_prep = mean_average_precision(results_l_prep)

print("MRR@10 : %.4f" % mrr_l_prep)
print("MAP    : %.4f" % map_l_prep)

# ─────────────────────────────────────────────────────────────────────────────
# 9.  SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("SUMMARY — Retrieval Effectiveness")
print("=" * 60)

summary = pd.DataFrame({
    "Model"        : ["BM25Okapi (raw)", "BM25Okapi (preprocessed)",
                      "BM25L (raw)",     "BM25L (preprocessed)"],
    "MRR@10"       : [mrr_okapi_raw, mrr_okapi_prep, mrr_l_raw, mrr_l_prep],
    "MAP"          : [map_okapi_raw, map_okapi_prep, map_l_raw, map_l_prep],
})
print(summary.to_string(index=False))

Read 1460 documents, 112 queries and 76 mappings from CISI dataset
Average 40.97 and 1 min relevant documents per query
Queries without relevant documents: [ 35  37  39  46  47  50  52  58  59  62  63  67  69  71  72  73  74  76
  77  79  82  84  85  86  87  88  90  92  93 102 104 105 106 107 109 111]

BM25Okapi — raw corpus (no preprocessing)
MRR@10 : 0.3146
MAP    : 0.0794

BM25Okapi — preprocessed corpus (lower + stopwords + stemming)
MRR@10 : 0.4187
MAP    : 0.1283

BM25L — raw corpus (no preprocessing)
MRR@10 : 0.2100
MAP    : 0.0664

BM25L — preprocessed corpus (lower + stopwords + stemming)
MRR@10 : 0.2678
MAP    : 0.0968

SUMMARY — Retrieval Effectiveness
                   Model   MRR@10      MAP
         BM25Okapi (raw) 0.314576 0.079387
BM25Okapi (preprocessed) 0.418683 0.128327
             BM25L (raw) 0.209970 0.066426
    BM25L (preprocessed) 0.267843 0.096756
